# INDIA-METAGENE v4 — T4 Optimized
## Fine-tuning METAGENE-1 on Gujarat Wastewater

### What changed from v3
- **No 8-bit quantization** — was causing 2 seq/s instead of 8 seq/s on T4
- **float16 model** — faster training, fits T4 with LoRA on 2 attention layers
- **50K sequences/epoch** — completes in ~1.7 hrs on T4 (not 69 hrs)
- **10 total epochs** — continuing from v2 epoch 3 checkpoint

### Expected timeline
- ~1.7 hrs per epoch on T4
- 7 remaining epochs (4-10) = ~12 hrs total
- 1-2 sessions on free Colab

### Session strategy
Each session: upload notebook → paste token → Run all → auto-resumes from checkpoint

### Before running
1. Runtime → Change runtime type → **T4 GPU**
2. Paste HF token in Cell 1
3. Runtime → Run all

In [ ]:
# Cell 1: Configuration — only cell you need to edit
HF_TOKEN   = 'PASTE_YOUR_HF_TOKEN_HERE'
HF_REPO    = 'saurabhthakar3/gujarat-wastewater-shotgun'
MODEL_REPO = 'saurabhthakar3/india-metagene-1'

import os
os.environ['HF_TOKEN'] = HF_TOKEN
print('Token set.')

In [ ]:
# Cell 2: Install dependencies
import subprocess, sys
pkgs = [
    'transformers>=4.40', 'accelerate>=0.27', 'safetensors',
    'sentencepiece', 'huggingface_hub', 'scipy', 'scikit-learn',
    'seaborn', 'torchao>=0.16.0', 'peft>=0.10.0',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + pkgs, check=True)
import torchao, peft
print(f'torchao: {torchao.__version__} | peft: {peft.__version__}')
print('Dependencies installed.')

In [ ]:
# Cell 3: GPU check
import torch
assert torch.cuda.is_available(), 'No GPU — enable GPU runtime'
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU : {gpu_name}')
print(f'VRAM: {vram_gb:.1f} GB')

if vram_gb > 35:
    GPU_MODE = 'A100'
    print('A100 detected.')
else:
    GPU_MODE = 'T4'
    print('T4 detected — running in float16 LoRA mode.')

In [ ]:
# Cell 4: Create model repo (safe to re-run)
from huggingface_hub import HfApi
api = HfApi()
try:
    api.create_repo(repo_id=MODEL_REPO, repo_type='model',
                    private=True, token=HF_TOKEN)
    print(f'Created: {MODEL_REPO}')
except Exception as e:
    print(f'Repo exists (OK).')

In [ ]:
# Cell 5: Download data + auto-restore latest checkpoint
import shutil, json
from pathlib import Path
from huggingface_hub import snapshot_download, HfApi, hf_hub_download

DATA_ROOT      = Path('/content/gujarat_shotgun')
FASTA_DIR      = DATA_ROOT / 'phase3_fasta'
METADATA_CSV   = DATA_ROOT / 'sample_metadata_complete.csv'
CHECKPOINT_DIR = Path('/content/checkpoints')
RESULTS_DIR    = Path('/content/finetune_results')
MODEL_CACHE    = Path('/content/model_cache')

for d in [DATA_ROOT, CHECKPOINT_DIR, RESULTS_DIR, MODEL_CACHE]:
    d.mkdir(exist_ok=True)

# Download FASTA data
print('Downloading FASTA data...')
snapshot_download(
    repo_id=HF_REPO, repo_type='dataset',
    local_dir=str(DATA_ROOT), token=HF_TOKEN,
    ignore_patterns=['results/*', 'finetune_results/*', 'finetune_results_v3/*'],
)
print('Data downloaded.')

# Auto-restore latest epoch checkpoint
print('\nChecking for saved checkpoints...')
api       = HfApi()
START_EPOCH = 0
RESUME_CKPT = None

try:
    files       = list(api.list_repo_files(repo_id=MODEL_REPO, repo_type='model', token=HF_TOKEN))
    ckpt_dirs   = set(f.split('/')[0] for f in files if f.startswith('checkpoint_lora_'))
    epoch_ckpts = sorted([d for d in ckpt_dirs if 'final' in d])
    if epoch_ckpts:
        latest = epoch_ckpts[-1]
        print(f'Restoring checkpoint: {latest}')
        snapshot_download(
            repo_id=MODEL_REPO, repo_type='model',
            local_dir=str(CHECKPOINT_DIR), token=HF_TOKEN,
            allow_patterns=[f'{latest}/*'],
        )
        RESUME_CKPT = CHECKPOINT_DIR / latest
        state_file  = RESUME_CKPT / 'training_state.json'
        if state_file.exists():
            state       = json.load(open(state_file))
            START_EPOCH = state['epoch']
            print(f'Resuming from epoch {START_EPOCH}')
    else:
        print('No checkpoints found — starting fresh.')
except Exception as e:
    print(f'Could not restore: {e}')

# Restore training history
history_path = RESULTS_DIR / 'training_history.json'
try:
    files = list(api.list_repo_files(repo_id=MODEL_REPO, repo_type='model', token=HF_TOKEN))
    if 'training_history.json' in files:
        hf_hub_download(
            repo_id=MODEL_REPO, repo_type='model',
            filename='training_history.json',
            local_dir=str(RESULTS_DIR), token=HF_TOKEN
        )
        print('Training history restored.')
except: pass

fastas = sorted(FASTA_DIR.glob('*.fasta.gz'))
print(f'\nFASTA files : {len(fastas)}')
print(f'Start epoch : {START_EPOCH}')
print(f'Resume ckpt : {RESUME_CKPT}')
assert len(fastas) > 0

In [ ]:
# Cell 6: Stratified train/val split (same seed as v2/v3 — reproducible)
import pandas as pd, random
random.seed(42)

meta_raw = pd.read_csv(METADATA_CSV)
fasta_df = pd.DataFrame({
    'fasta_path': fastas,
    'fasta_id'  : [f.stem.replace('.fasta','') for f in fastas],
    'meta_key'  : [f.stem.replace('.fasta','').replace('_R1','') for f in fastas],
})
fasta_df = fasta_df.merge(meta_raw, left_on='meta_key', right_on='sample_id', how='left')

CITIES      = sorted(fasta_df['city'].dropna().unique())
SEASONS     = ['Summer','Monsoon','PreWinter','Winter']
CITY_COLORS = {'Ahmedabad':'#1f77b4','Gandhinagar':'#ff7f0e',
               'Rajkot':'#2ca02c','Vadodara':'#d62728'}
SEASON_ORDER = ['Summer','Monsoon','PreWinter','Winter']

val_ids = []
for city in CITIES:
    for season in SEASONS:
        subset = fasta_df[
            (fasta_df['city']==city) & (fasta_df['season']==season)
        ]['fasta_id'].tolist()
        if subset:
            val_ids.append(random.choice(subset))

train_fastas = [f for f in fastas if f.stem.replace('.fasta','') not in val_ids]
val_fastas   = [f for f in fastas if f.stem.replace('.fasta','') in val_ids]

print(f'Train: {len(train_fastas)} | Val: {len(val_fastas)} (1 per city × season)')

In [ ]:
# Cell 7: Load METAGENE-1 in float16 + LoRA (fast, fits T4)
# NO 8-bit quantization — it was causing 2 seq/s, now expect 8 seq/s
import torch, time
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

device   = torch.device('cuda')
MODEL_ID = 'metagene-ai/METAGENE-1'

print(f'Loading {MODEL_ID} in float16...')
t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=str(MODEL_CACHE))

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    cache_dir=str(MODEL_CACHE),
    torch_dtype=torch.float16,
    device_map='cuda',
)

# Apply LoRA or resume from checkpoint
if RESUME_CKPT and RESUME_CKPT.exists():
    print(f'Loading LoRA checkpoint: {RESUME_CKPT}')
    model = PeftModel.from_pretrained(model, str(RESUME_CKPT), is_trainable=True)
else:
    print('Applying fresh LoRA adapters...')
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=['q_proj','v_proj'],  # 2 layers only — saves VRAM on T4
        bias='none',
    )
    model = get_peft_model(model, lora_config)

model.print_trainable_parameters()
vram = torch.cuda.memory_allocated()/1e9
total_vram = torch.cuda.get_device_properties(0).total_memory/1e9
print(f'\nLoaded in {time.time()-t0:.0f}s')
print(f'VRAM used : {vram:.1f} / {total_vram:.1f} GB')
print(f'VRAM free : {total_vram-vram:.1f} GB')

if total_vram - vram < 1.5:
    print('WARNING: Less than 1.5GB free — may OOM during training.')
    print('If OOM occurs, reduce BATCH_SIZE to 1 in Cell 9.')
else:
    print('VRAM OK — ready to train.')

In [ ]:
# Cell 8: Cast LoRA params to float32 — CRITICAL for learning
# Without this gradients underflow in float16 and val loss stays flat
import torch

n_fixed = 0
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()
        n_fixed += 1

trainable_dtypes = set(p.dtype for p in model.parameters() if p.requires_grad)
frozen_dtypes    = set(p.dtype for p in model.parameters() if not p.requires_grad)

print(f'Fixed {n_fixed} LoRA tensors → float32')
print(f'Trainable: {trainable_dtypes}')  # must be float32
print(f'Frozen   : {frozen_dtypes}')     # float16
print(f'VRAM     : {torch.cuda.memory_allocated()/1e9:.1f} GB')
assert torch.float32 in trainable_dtypes, 'ERROR: LoRA not float32!'
print('Float32 fix applied.')

In [ ]:
# Cell 9: Helpers + training config
import gzip, random, json
import numpy as np
from huggingface_hub import HfApi
from torch.amp import autocast, GradScaler

SEED               = 42
MAX_SEQ_LEN        = 512
MAX_EPOCHS         = 10
SAVE_EVERY_N_STEPS = 200
LR                 = 1e-5

# T4 settings — tuned for speed
BATCH_SIZE       = 2    # keep low to avoid OOM
GRAD_ACCUM       = 16   # effective batch = 32
N_SEQS_PER_EPOCH = 50_000  # 1.7 hrs/epoch on T4

random.seed(SEED); np.random.seed(SEED)


def sample_sequences(fasta_list, n_total, seed=SEED):
    """Stream-sample sequences — never loads all into RAM."""
    rng        = random.Random(seed)
    result     = []
    n_per_file = max(1, n_total // len(fasta_list))
    shuffled   = list(fasta_list); rng.shuffle(shuffled)
    for fasta in shuffled:
        opener = gzip.open if str(fasta).endswith('.gz') else open
        seqs   = []
        with opener(fasta, 'rt') as f:
            seq = []
            for line in f:
                line = line.rstrip()
                if line.startswith('>'):
                    if seq: seqs.append(''.join(seq))
                    seq = []
                    if len(seqs) >= n_per_file: break
                else: seq.append(line)
        result.extend(seqs[:n_per_file])
        if len(result) >= n_total: break
    rng.shuffle(result)
    return result[:n_total]


def make_batches(seqs, bs=BATCH_SIZE):
    for i in range(0, len(seqs), bs):
        yield seqs[i:i+bs]


@torch.no_grad()
def evaluate_loss(fasta_list, n_seqs=500):
    model.eval()
    seqs   = sample_sequences(fasta_list, n_seqs)
    losses = []
    for batch in make_batches(seqs, bs=4):
        inputs = tokenizer(
            batch, return_tensors='pt', padding=True,
            truncation=True, max_length=MAX_SEQ_LEN,
            add_special_tokens=False
        ).to(device)
        with autocast('cuda', dtype=torch.float16):
            outputs = model(**inputs, labels=inputs['input_ids'])
        sl = outputs.logits[...,:-1,:].contiguous().float()
        tl = inputs['input_ids'][...,1:].contiguous()
        sm = inputs['attention_mask'][...,1:].contiguous().float()
        loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
        tok_loss = loss_fct(sl.view(-1,sl.size(-1)),tl.view(-1)).view(tl.size())
        norm_loss = (tok_loss*sm).sum(1)/sm.sum(1).clamp(min=1)
        losses.extend(norm_loss.cpu().float().numpy().tolist())
        del outputs, inputs
        torch.cuda.empty_cache()
    model.train()
    return float(np.mean(losses)), float(np.std(losses))


def save_checkpoint(epoch, step, train_loss, val_loss):
    ckpt_name = f'lora_epoch{epoch}_step{step}'
    ckpt_path = CHECKPOINT_DIR / ckpt_name
    ckpt_path.mkdir(exist_ok=True)
    model.save_pretrained(str(ckpt_path))
    tokenizer.save_pretrained(str(ckpt_path))
    json.dump(
        {'epoch':epoch,'step':step,'train_loss':train_loss,'val_loss':val_loss},
        open(ckpt_path/'training_state.json','w'), indent=2
    )
    print(f'  Uploading {ckpt_name}...', end=' ')
    try:
        HfApi().upload_folder(
            folder_path=str(ckpt_path),
            repo_id=MODEL_REPO, repo_type='model',
            path_in_repo=f'checkpoint_{ckpt_name}',
            token=HF_TOKEN,
        )
        print('✓')
    except Exception as e:
        print(f'failed: {e}')


steps_per_epoch = N_SEQS_PER_EPOCH // BATCH_SIZE // GRAD_ACCUM
est_hrs         = steps_per_epoch * GRAD_ACCUM * BATCH_SIZE / 8 / 3600

print('Helpers ready.')
print(f'Sequences/epoch  : {N_SEQS_PER_EPOCH:,}')
print(f'Effective batch  : {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM}')
print(f'Steps/epoch      : {steps_per_epoch:,}')
print(f'Est. hrs/epoch   : {est_hrs:.1f} hrs on T4 at 8 seq/s')
print(f'Remaining epochs : {MAX_EPOCHS - START_EPOCH}')
print(f'Est. total time  : {est_hrs*(MAX_EPOCHS-START_EPOCH):.1f} hrs')

In [ ]:
# Cell 10: Baseline — restore from HF or compute fresh
import json, time
from huggingface_hub import HfApi, hf_hub_download

baseline_path = RESULTS_DIR / 'baseline.json'

if baseline_path.exists():
    baseline          = json.load(open(baseline_path))
    baseline_val_mean = baseline['val_mean']
    print(f'Baseline from file: {baseline_val_mean:.4f}')
else:
    try:
        hf_hub_download(
            repo_id=MODEL_REPO, repo_type='model',
            filename='baseline.json',
            local_dir=str(RESULTS_DIR), token=HF_TOKEN
        )
        baseline          = json.load(open(baseline_path))
        baseline_val_mean = baseline['val_mean']
        print(f'Baseline from HuggingFace: {baseline_val_mean:.4f}')
    except:
        print('Computing baseline...')
        t0 = time.time()
        baseline_val_mean, baseline_val_std = evaluate_loss(val_fastas, n_seqs=1000)
        baseline = {
            'val_mean'   : baseline_val_mean,
            'val_std'    : baseline_val_std,
            'paper_ww'   : 1.24,
            'paper_human': 5.22,
        }
        json.dump(baseline, open(baseline_path,'w'), indent=2)
        try:
            HfApi().upload_file(
                path_or_fileobj=str(baseline_path),
                path_in_repo='baseline.json',
                repo_id=MODEL_REPO, repo_type='model', token=HF_TOKEN
            )
        except: pass
        print(f'Baseline: {baseline_val_mean:.4f} ({time.time()-t0:.0f}s)')

print(f'\nBaseline (original METAGENE-1) : {baseline_val_mean:.4f}')
print(f'v2 best val loss (epoch 3)     : 4.8311')
print(f'Paper WW in-dist reference     : 1.2400')
print(f'Target after 10 epochs         : ~4.70')

In [ ]:
# Cell 11: Fine-tuning loop — disconnect-proof, auto-resumes
import torch, time, json, random
import numpy as np
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from transformers import get_linear_schedule_with_warmup
from huggingface_hub import HfApi

# Load or init history
history_path = RESULTS_DIR / 'training_history.json'
if history_path.exists():
    history = json.load(open(history_path))
    print(f'History: {len(history["epoch_val_loss"])} epochs done')
    if history['epoch_val_loss']:
        print(f'Best val loss : {min(history["epoch_val_loss"]):.4f}')
        print(f'Last val loss : {history["epoch_val_loss"][-1]:.4f}')
else:
    history = {
        'epoch_train_loss': [], 'epoch_val_loss': [],
        'step_losses': [], 'epochs_done': []
    }

# Optimizer on float32 LoRA params only
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_params, lr=LR, weight_decay=0.01)
scaler    = GradScaler('cuda')

steps_per_epoch = N_SEQS_PER_EPOCH // BATCH_SIZE // GRAD_ACCUM
remaining       = MAX_EPOCHS - START_EPOCH
total_steps     = steps_per_epoch * remaining
warmup_steps    = min(50, total_steps // 10)
scheduler       = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f'Starting from epoch {START_EPOCH+1} / {MAX_EPOCHS}')
print(f'Sequences/epoch : {N_SEQS_PER_EPOCH:,}')
print(f'Steps/epoch     : {steps_per_epoch:,}')
print(f'Save every      : {SAVE_EVERY_N_STEPS} steps (~{SAVE_EVERY_N_STEPS*BATCH_SIZE*GRAD_ACCUM:,} seqs)')
print('─'*70)

for epoch in range(START_EPOCH, MAX_EPOCHS):
    print(f'\n=== EPOCH {epoch+1}/{MAX_EPOCHS} ===')
    t_epoch = time.time()

    print('Sampling sequences...')
    epoch_seqs = sample_sequences(train_fastas, N_SEQS_PER_EPOCH, seed=SEED+epoch)
    print(f'Sampled {len(epoch_seqs):,}')

    epoch_losses = []
    accum_loss   = 0.0
    global_step  = 0
    optimizer.zero_grad()
    model.train()

    for step, batch in enumerate(make_batches(epoch_seqs)):
        inputs = tokenizer(
            batch, return_tensors='pt', padding=True,
            truncation=True, max_length=MAX_SEQ_LEN,
            add_special_tokens=False
        ).to(device)

        with autocast('cuda', dtype=torch.float16):
            outputs = model(**inputs, labels=inputs['input_ids'])
            loss    = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        accum_loss += loss.item()
        del outputs, inputs
        torch.cuda.empty_cache()

        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

            step_loss = accum_loss
            epoch_losses.append(step_loss)
            history['step_losses'].append(step_loss)
            accum_loss  = 0.0
            global_step += 1

            if global_step % 50 == 0:
                elapsed = time.time() - t_epoch
                rate    = (step+1)*BATCH_SIZE / elapsed
                pct     = (step+1) / len(epoch_seqs) * BATCH_SIZE * 100
                eta_min = (len(epoch_seqs)/BATCH_SIZE-step-1)/max(rate,1)*BATCH_SIZE/60
                print(f'  Step {global_step:>4} | '
                      f'loss={step_loss:.4f} | '
                      f'{pct:.1f}% | '
                      f'{rate:.0f} seq/s | '
                      f'ETA {eta_min:.0f} min')

            if global_step % SAVE_EVERY_N_STEPS == 0:
                print(f'\n  --- Checkpoint step {global_step} ---')
                val_mean, val_std = evaluate_loss(val_fastas, n_seqs=500)
                improvement = baseline_val_mean - val_mean
                print(f'  Val loss : {val_mean:.4f} ± {val_std:.4f}')
                print(f'  Baseline : {baseline_val_mean:.4f}  '
                      f'Δ={improvement:+.4f}')
                save_checkpoint(epoch+1, global_step,
                                float(np.mean(epoch_losses[-50:])), val_mean)
                json.dump(history, open(history_path,'w'), indent=2)
                try:
                    HfApi().upload_file(
                        path_or_fileobj=str(history_path),
                        path_in_repo='training_history.json',
                        repo_id=MODEL_REPO, repo_type='model', token=HF_TOKEN)
                except: pass
                model.train()

    # End of epoch
    epoch_train_loss = float(np.mean(epoch_losses))
    elapsed_min = (time.time()-t_epoch)/60
    print(f'\nEpoch {epoch+1} done | {elapsed_min:.0f} min')
    print(f'  Train loss : {epoch_train_loss:.4f}')

    val_mean, val_std = evaluate_loss(val_fastas, n_seqs=1000)
    improvement = baseline_val_mean - val_mean
    print(f'  Val loss   : {val_mean:.4f} ± {val_std:.4f}')
    print(f'  Baseline   : {baseline_val_mean:.4f}')
    print(f'  Improvement: {improvement:+.4f} ({improvement/baseline_val_mean*100:.2f}%)')

    history['epoch_train_loss'].append(epoch_train_loss)
    history['epoch_val_loss'].append(val_mean)
    history['epochs_done'].append(epoch+1)
    json.dump(history, open(history_path,'w'), indent=2)

    save_checkpoint(epoch+1, 'final', epoch_train_loss, val_mean)
    try:
        HfApi().upload_file(
            path_or_fileobj=str(history_path),
            path_in_repo='training_history.json',
            repo_id=MODEL_REPO, repo_type='model', token=HF_TOKEN)
    except: pass

    START_EPOCH = epoch + 1

print('\n' + '='*65)
print('ALL EPOCHS COMPLETE')
print(f'Final val loss   : {history["epoch_val_loss"][-1]:.4f}')
print(f'Baseline         : {baseline_val_mean:.4f}')
print(f'Total improvement: {baseline_val_mean-history["epoch_val_loss"][-1]:+.4f}')
print('='*65)

In [ ]:
# Cell 12: Training curves
import matplotlib.pyplot as plt
import numpy as np, json

history  = json.load(open(RESULTS_DIR/'training_history.json'))
baseline = json.load(open(RESULTS_DIR/'baseline.json'))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
sl = history['step_losses']
w  = min(100, max(1, len(sl)//20))
if len(sl) >= w:
    sm = np.convolve(sl, np.ones(w)/w, mode='valid')
    ax.plot(sl, alpha=0.15, color='steelblue')
    ax.plot(range(w-1,len(sl)), sm, color='steelblue', lw=2, label='Smoothed')
ax.axhline(baseline['val_mean'], color='red', ls='--', lw=1.5,
           label=f'Baseline ({baseline["val_mean"]:.3f})')
ax.axhline(baseline['paper_ww'], color='green', ls='--', lw=1.5,
           label=f'Paper WW ({baseline["paper_ww"]})')
ax.set_xlabel('Step'); ax.set_ylabel('CE Loss')
ax.set_title('Training Loss Curve', fontweight='bold')
ax.legend(fontsize=8)

ax = axes[1]
epochs     = [0] + history['epochs_done']
val_losses = [baseline['val_mean']] + history['epoch_val_loss']
ax.plot(epochs, val_losses, 'o-', color='steelblue', lw=2, ms=8, label='Val loss')
ax.axhline(baseline['val_mean'], color='red', ls='--', lw=1.5,
           label=f'Baseline ({baseline["val_mean"]:.3f})')
ax.axhline(baseline['paper_ww'], color='green', ls='--', lw=1.5,
           label=f'Paper WW ({baseline["paper_ww"]})')
ax.axhline(baseline['paper_human'], color='orange', ls='--', lw=1.5,
           label=f'Human genome ({baseline["paper_human"]})')
ax.set_xlabel('Epoch'); ax.set_ylabel('Val CE Loss')
ax.set_title('Validation Loss per Epoch', fontweight='bold')
ax.set_xticks(epochs); ax.legend(fontsize=8)

plt.suptitle('INDIA-METAGENE v4: Fine-tuning on Gujarat Wastewater',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR/'training_curve_v4.pdf', dpi=150, bbox_inches='tight')
plt.savefig(RESULTS_DIR/'training_curve_v4.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Epochs done : {history["epochs_done"]}')
print(f'Val losses  : {[round(v,4) for v in history["epoch_val_loss"]]}')

In [ ]:
# Cell 13: Per-sample evaluation — before vs after
import torch, json, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

baseline = json.load(open(RESULTS_DIR/'baseline.json'))

print('Per-sample val loss (fine-tuned)...')
per_sample = []
for fasta in val_fastas:
    sid      = fasta.stem.replace('.fasta','')
    meta_key = sid.replace('_R1','')
    row      = meta_raw[meta_raw['sample_id']==meta_key]
    city     = row['city'].iloc[0]   if len(row) else 'Unknown'
    season   = row['season'].iloc[0] if len(row) else 'Unknown'
    mean_loss, std_loss = evaluate_loss([fasta], n_seqs=300)
    per_sample.append({
        'sample_id'     : sid, 'city': city, 'season': season,
        'finetuned_loss': mean_loss,
        'baseline_loss' : baseline['val_mean'],
        'improvement'   : baseline['val_mean'] - mean_loss,
    })
    print(f'  {sid:<15} {city:<12} {season:<10} '
          f'{baseline["val_mean"]:.3f} → {mean_loss:.3f} '
          f'(Δ={baseline["val_mean"]-mean_loss:+.3f})')

results_df = pd.DataFrame(per_sample)
results_df.to_csv(RESULTS_DIR/'finetuned_val_results_v4.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
x  = range(len(results_df))
ax.bar(x, results_df['baseline_loss'], alpha=0.35, color='red', label='Baseline')
ax.bar(x, results_df['finetuned_loss'],
       color=[CITY_COLORS.get(c,'grey') for c in results_df['city']],
       alpha=0.85, label='Fine-tuned (v4)')
ax.axhline(1.24, color='green', ls='--', lw=1.5, label='Paper WW (1.24)')
ax.set_xticks(x); ax.set_xticklabels(results_df['sample_id'], rotation=90, fontsize=7)
ax.set_ylabel('Mean CE Loss')
ax.set_title('Before vs After Fine-tuning', fontweight='bold'); ax.legend(fontsize=7)

ax = axes[1]
s_ord = [s for s in SEASON_ORDER if s in results_df['season'].values]
sns.boxplot(data=results_df, x='season', y='finetuned_loss',
            order=s_ord, hue='season', palette='Set1', ax=ax, legend=False)
ax.axhline(baseline['val_mean'], color='red', ls='--', lw=1.5,
           label=f'Baseline ({baseline["val_mean"]:.3f})')
ax.axhline(1.24, color='green', ls='--', lw=1.5, label='Paper WW')
ax.set_title('Fine-tuned Loss by Season', fontweight='bold')
ax.set_ylabel('Mean CE Loss'); ax.legend(fontsize=8)

plt.suptitle('INDIA-METAGENE v4: Results', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR/'finetuning_results_v4.pdf', dpi=150, bbox_inches='tight')
plt.savefig(RESULTS_DIR/'finetuning_results_v4.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== SUMMARY ===')
print(f'Baseline         : {baseline["val_mean"]:.4f}')
print(f'Fine-tuned       : {results_df["finetuned_loss"].mean():.4f} '
      f'± {results_df["finetuned_loss"].std():.4f}')
print(f'Mean improvement : {results_df["improvement"].mean():+.4f}')
print(f'Paper WW ref     : 1.2400')

In [ ]:
# Cell 14: Save all results to HuggingFace
from huggingface_hub import HfApi
api = HfApi()
print('Saving to HuggingFace...')
for f in RESULTS_DIR.glob('*'):
    try:
        api.upload_file(
            path_or_fileobj=str(f),
            path_in_repo=f'finetune_results_v4/{f.name}',
            repo_id=HF_REPO, repo_type='dataset', token=HF_TOKEN
        )
        print(f'  ✓ {f.name}')
    except Exception as e:
        print(f'  ✗ {f.name}: {e}')
print('Done.')